<a href="https://colab.research.google.com/github/sassial/AITextGenerationDetection/blob/main/ProjetIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Overview: AI-Generated Text Detection

With the rapid advancement of large language models (LLMs), distinguishing between human-written and AI-generated content has become a critical challenge. In this project, we designed and implemented a system to perform this classification task using modern Natural Language Processing (NLP) techniques and various machine learning algorithms.

### Project Objectives and Accomplishments
* **Exploration:** We analyzed the unique characteristics of AI-generated versus human-written text.
* **Preprocessing:** We implemented cleaning and normalization pipelines to prepare textual data for modeling.
* **Text Representation:** We benchmarked multiple embedding techniques, including Word2Vec, Doc2Vec, GloVe, and BERT.
* **Modeling:** We trained and evaluated five different classification models (KNN, SVM, Decision Tree, Random Forest, and MLP).
* **Comparative Analysis:** We quantified the effectiveness of each approach to identify the most robust solution for detecting AI-generated text.

## Downloads

In [ ]:
!pip install nltk spacy gensim transformers torch

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

!python -m spacy download en_core_web_sm

!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
import string
import os
import uuid
import gc
import torch
import urllib.request
import zipfile
from collections import Counter
from tqdm import tqdm

# NLP and Embeddings
import spacy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.scripts.glove2word2vec import glove2word2vec
from transformers import BertTokenizer, BertModel

# Scikit-learn utilities
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score, accuracy_score, precision_recall_fscore_support
from sklearn.decomposition import PCA

# Download and setup GloVe directory
if not os.path.exists("glove"):
    os.makedirs("glove")
    with zipfile.ZipFile("glove.6B.zip", 'r') as zip_ref:
        zip_ref.extractall("glove")

# Dataset Summary

For this analysis, we utilized the *LLM Detect AI Generated Text* dataset. This collection contains labeled examples indicating content origin, providing the foundation for our binary classification task. The dataset encompasses diverse writing styles and topics, allowing us to test the generalization capabilities of our models.

## Importing the Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/ColabNotebooks/ProjetIA/'

## Loading

In [ ]:
tr_e = pd.read_csv(path + 'train_essays.csv')
t_p = pd.read_csv(path + 'train_prompts.csv')
te_e = pd.read_csv(path + 'test_essays.csv')
s_s = pd.read_csv(path + 'sample_submission.csv')
l_e = pd.read_csv(path + 'llm_essays.csv')

## Inspecting the Dataset

In [ ]:
tr_e.head()

In [ ]:
print("Missing Values:")
print(tr_e.isnull().sum())

In [ ]:
print("Duplicates:", tr_e.duplicated().sum())

In [ ]:
l_e.head()

In [ ]:
print("Missing Values:")
print(l_e.isnull().sum())

In [ ]:
# Get all existing IDs from tr_e for uniqueness check
existing_tr_e_ids = set(tr_e['id'].astype(str))

# List to store new unique IDs for l_e
new_l_e_ids = []
generated_l_e_ids = set()

print(f"Generating {len(l_e)} unique IDs for l_e...")
for i in range(len(l_e)):
    new_id = None
    attempts = 0
    max_attempts = 1000
    while True:
        potential_id = '00' + uuid.uuid4().hex[2:8]
        if potential_id not in existing_tr_e_ids and potential_id not in generated_l_e_ids:
            new_id = potential_id
            generated_l_e_ids.add(new_id)
            break
        attempts += 1
        if attempts > max_attempts:
            new_id = potential_id
            break
    new_l_e_ids.append(new_id)

l_e['id'] = new_l_e_ids
print("Verification: All new l_e IDs assigned.")

In [ ]:
l_e['generated']=1

In [ ]:
l_e.head()

In [ ]:
l_e.to_csv(path + 'llm_essays.csv', index=False)

In [ ]:
tr_e = pd.concat([tr_e, l_e], ignore_index=True)
tr_e.drop_duplicates(subset=['text'], inplace=True)

## Analyzing Dataset

### Size:

In [ ]:
print("Dataset shape:", tr_e.shape)
print("Total essays:", len(tr_e))
print("Memory usage (MB):", tr_e.memory_usage(deep=True).sum() / 1024**2)

### Balance:

#### Human vs LLM:

In [ ]:
print(pd.concat([tr_e['generated'].value_counts(),
                 tr_e['generated'].value_counts(normalize=True).mul(100)],
                axis=1,
                keys=['count', 'percentage']))

#### Prompt 1 vs Prompt 2:

In [ ]:
print(pd.concat([tr_e['prompt_id'].value_counts(),
                 tr_e['prompt_id'].value_counts(normalize=True).mul(100)],
                axis=1,
                keys=['count', 'percentage']))

In [ ]:
pd.crosstab(tr_e['prompt_id'], tr_e['generated'])

### Diversity

#### Number of Characters:

In [ ]:
tr_e['char_length'] = tr_e['text'].apply(len)
print(tr_e.groupby('generated')['char_length'].describe())

In [ ]:
sns.histplot(
    data=tr_e,
    x='char_length',
    hue='generated',
    bins=50,
    kde=True
)

plt.title("Essay Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.show()

## Potential biases and inconsistencies

* Size: 2750 essays isn’t a lot of data vectors for machine learning; a more appropriate amount would be at least 10000.

* Balance: There’s 1375 LLM generated essays to 1375 human written essays, which is quite balanced, but at least 1372 of the 1375 llm generated essays are generated by ChatGPT, limiting the scope of our LLM identification.

* Diversity: There’s a lot of diversity in essay length

# Text Processing and Normalization

To transform raw text into a format suitable for machine learning, we applied a comprehensive preprocessing pipeline consisting of the following steps:

1. **Cleaning:** Removed punctuation, special characters, and numerical noise.
2. **Lowercasing:** Standardized all text to lowercase for uniformity.
3. **Tokenization:** Segmented text into individual word tokens.
4. **Stopword Removal:** Eliminated common non-informative words to reduce noise.
5. **Lemmatization:** Reduced words to their base linguistic forms.
6. **Exporting:** Saved the finalized preprocessed dataset for consistent use across all model experiments.

## Step 1: Cleaning

In [ ]:
def clean_text(text):

    # Convert to string
    text = str(text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove special characters and noise
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# removing punctuation, special characters, and noise
tr_e['clean_text'] = tr_e['text'].apply(clean_text)
tr_e[['text', 'clean_text']].head()

## Step 2: Lowercasing

In [ ]:
# ensuring uniformity
tr_e['lower_text'] = tr_e['clean_text'].str.lower()
tr_e[['text', 'lower_text']].head()

## Step 3: Tokenization

In [ ]:
# splitting text into words or subwords
tr_e['tokens'] = tr_e['lower_text'].apply(word_tokenize)
tr_e[['text', 'tokens']].head()

## Step 4: Stopword Removal

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    filtered = [word for word in tokens if word not in stop_words]
    return filtered

# removing common non-informative words
tr_e['filtered_tokens'] = tr_e['tokens'].apply(remove_stopwords)
tr_e[['text', 'filtered_tokens']].head()

## Step 5: Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]
    return lemmatized

# reducing words to their base form
tr_e['lemmatized_tokens'] = tr_e['filtered_tokens'].apply(lemmatize_tokens)
tr_e[['text', 'lemmatized_tokens']].head()

## Step 6: Tokens to Text

In [ ]:
# Converting tokens back into text
tr_e['preprocessed_text'] = tr_e['lemmatized_tokens'].apply(lambda tokens: ' '.join(tokens))
tr_e[['text', 'preprocessed_text']].head()

## Step 7: Save Dataset

In [ ]:
# saving the final data structure with all the preprocessed data into a separate .csv file
tr_e.to_csv(path + 'preprocessed_dataset.csv', index=False)

# Exploratory Data Analysis

We conducted a thorough exploration of the dataset to identify structural patterns and potential challenges. Our analysis focused on:

* **Class Balance:** Verifying the distribution of human vs. AI samples.
* **Text Metrics:** Comparing document lengths and vocabulary richness between the two classes.
* **Lexical Signatures:** Identifying the most frequent terms that distinguish AI generation from human writing.

Our findings indicate that AI-generated text in this dataset exhibits specific repetitive structures and distinct lexical distributions, which we leveraged for classification.

## Step 1: Inspection

In [ ]:
# Dataset shape
print("Dataset Shape:", tr_e.shape)

# Dataset information
print("\nDataset Info:")
print(tr_e.info())

## Step 2: Class Distribution

In [ ]:
print("\nHuman vs LLM:")
combined = pd.DataFrame({
    'Count': tr_e['generated'].value_counts(),
    'Percentage': tr_e['generated'].value_counts(normalize=True) * 100
})
print(combined)

print("\nClass Distribution Bar Graph")
tr_e['generated'].value_counts().plot(kind='bar')

plt.xticks([0,1], ['Human', 'LLM'])
plt.ylabel("Number of Essays")
plt.title("Class Distribution")
plt.show()

## Step 3: Text Length Distribution

In [ ]:
tr_e['text_length'] = tr_e['tokens'].apply(len)
print("\nWord Count Statistics")
print(tr_e['text_length'].describe())

print("\nText Length Distribution")
plt.figure(figsize=(10,6))

sns.histplot(
    data=tr_e,
    x='text_length',
    hue='generated',
    bins=40,
    kde=True
)

plt.title("Text Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

In [ ]:
avg_lengths = tr_e.groupby('generated')['text_length'].mean()
print("\nAverage Text Length by Class:")
print(avg_lengths)

## Step 4: Vocabulary Richness Analysis

In [ ]:
def vocabulary_richness(tokens):

    if len(tokens) == 0:
        return 0

    unique_words = len(set(tokens))
    total_words = len(tokens)

    return unique_words / total_words

tr_e['vocabulary_richness'] = tr_e['lemmatized_tokens'].apply(vocabulary_richness)
print("\nVocabulary Richness Statistics")
print(tr_e.groupby('generated')['vocabulary_richness'].describe())

In [ ]:
print("\nVocabulary Richness Distribution")
plt.figure(figsize=(10,6))

sns.boxplot(
    x='generated',
    y='vocabulary_richness',
    data=tr_e
)

plt.xticks([0,1], ['Human', 'AI'])

plt.title("Vocabulary Richness Comparison")

plt.show()

## Step 5: Most Frequent Word Analysis

In [ ]:
def get_average_word_freq(df, column='lemmatized_tokens', top_n=20):
    word_tf_sum = Counter()
    doc_count = len(df)

    if doc_count == 0:
        return {}

    for _, row in df.iterrows():
        tokens = row[column]
        if not tokens: # Handle empty token lists
            continue

        doc_word_count = len(tokens)
        doc_tf = Counter(tokens)

        for word, count in doc_tf.items():
            # Sum up Term Frequencies (TF) for each word across all documents
            word_tf_sum[word] += (count / doc_word_count)

    # Calculate the average Term Frequency by dividing by the number of documents
    avg_tf = {word: tf_sum / doc_count for word, tf_sum in word_tf_sum.items()}

    # Get top N by average TF
    top_words = sorted(avg_tf.items(), key=lambda x: x[1], reverse=True)[:top_n]

    return dict(top_words)

# Create comparison tables and plots
for prompt in [0, 1]:
    # Get data for human and AI
    human_data = tr_e[(tr_e['prompt_id'] == prompt) & (tr_e['generated'] == 0)]
    ai_data = tr_e[(tr_e['prompt_id'] == prompt) & (tr_e['generated'] == 1)]

    # Get average frequencies as dictionaries
    human_tf = get_average_word_freq(human_data, column='lemmatized_tokens')
    ai_tf = get_average_word_freq(ai_data, column='lemmatized_tokens')

    # Create a combined DataFrame with ALL unique words from both human and AI
    all_unique_words = set(list(human_tf.keys()) + list(ai_tf.keys()))
    comparison_df = pd.DataFrame({
        'Word': list(all_unique_words),
        'Human_TF': [human_tf.get(word, 0) for word in all_unique_words],
        'AI_TF': [ai_tf.get(word, 0) for word in all_unique_words]
    })

    # Sort by combined TF to show most important words from either group
    comparison_df['Max_TF'] = comparison_df[['Human_TF', 'AI_TF']].max(axis=1)
    comparison_df = comparison_df.sort_values('Max_TF', ascending=False).head(20)
    comparison_df = comparison_df.drop('Max_TF', axis=1)

    print(f"\n{'='*70}")
    print(f"Prompt {prompt} - Human vs AI Word Term Frequency Comparison")
    print(f"{'='*70}")
    print(comparison_df.to_string(index=False))

    # Create a single bar graph with two colors for Human vs AI
    fig, ax = plt.subplots(figsize=(12, 8))

    # Set up the positions for the bars
    y_pos = range(len(comparison_df['Word']))
    bar_height = 0.35

    # Plot horizontal bars
    ax.barh([i - bar_height/2 for i in y_pos], comparison_df['Human_TF'],
            height=bar_height, label='Human', color='skyblue', alpha=0.8)
    ax.barh([i + bar_height/2 for i in y_pos], comparison_df['AI_TF'],
            height=bar_height, label='AI', color='salmon', alpha=0.8)

    # Customize the plot
    ax.set_yticks(y_pos)
    ax.set_yticklabels(comparison_df['Word'])
    ax.set_xlabel('Average Term Frequency per Essay')
    ax.set_title(f'Prompt {prompt} - Most Frequent Words (Average TF): Human vs AI')
    ax.legend()
    ax.invert_yaxis()  # Invert y-axis to show highest frequency at the top

    plt.tight_layout()
    plt.show()

# Feature Extraction: Statistical Methods

We initially explored frequency-based representations to establish a baseline for the classification task:

* **Bag-of-Words (BoW):** A simple numerical representation based on word counts.
* **TF-IDF:** A weighted representation that emphasizes distinctive words while de-emphasizing common terms.

For the final model benchmarking, we focused on the cleaned text to preserve as much semantic context as possible.

In [ ]:
# Define features and labels
X_text = tr_e['lower_text']
y = tr_e['generated']

# Split the dataset before feature extraction to avoid data leakage
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Bag-of-Words

BoW converts text into numerical vectors based on word frequency.

In [ ]:
bow_vectorizer = CountVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_bow = bow_vectorizer.fit_transform(X_train_text)

X_test_bow = bow_vectorizer.transform(X_test_text)

print("BoW Training Shape:", X_train_bow.shape)
print("BoW Testing Shape:", X_test_bow.shape)

print(bow_vectorizer.get_feature_names_out()[:20])

### TF-IDF

TF-IDF improves upon BoW by reducing the importance of very common words and increasing the importance of distinctive words.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("TF-IDF Training Shape:", X_train_tfidf.shape)
print("TF-IDF Testing Shape:", X_test_tfidf.shape)

print(tfidf_vectorizer.get_feature_names_out()[:20])

# Advanced Text Embeddings

We implemented and compared several dense vector representation methods to capture semantic relationships:

* **Word2Vec:** Captures local context using a shallow neural network.
* **Doc2Vec:** Extends the Word2Vec concept to represent entire documents.
* **GloVe:** Leverages global word co-occurrence statistics from pre-trained embeddings.
* **BERT:** A transformer-based model providing deep contextualized representations.

Each embedding was evaluated based on its computational overhead and its impact on final classification performance.

## Word2Vec

In [ ]:
print("\nGenerating Word2Vec Embeddings...")

# Train Word2Vec model
word2vec_model = Word2Vec(
    sentences=tr_e['tokens'].tolist(),
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10
)

# Function to get document embedding by averaging word embeddings
def get_word2vec_embedding(tokens, model, vector_size):
    embeddings = []
    for word in tokens:
        if word in model.wv:
            embeddings.append(model.wv[word])
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(vector_size)

tr_e['word2vec_embeddings'] = tr_e['tokens'].apply(
    lambda tokens: get_word2vec_embedding(tokens, word2vec_model, 100)
)

print("\nWord2Vec Embeddings generated. Shape of first embedding:", tr_e['word2vec_embeddings'].iloc[0].shape)
print("\nShape of all Word2Vec embeddings:", np.stack(tr_e['word2vec_embeddings']).shape)
print("\nFirst 5 Word2Vec embeddings:\n", tr_e['word2vec_embeddings'].head())

### Word2Vec Evaluation

*   **Representation Quality**: Captures semantic relationships based on local context windows. Since it averages word vectors for the document, it loses word order but retains strong topical signal.
*   **Computational Cost**: Very low. Training on this dataset size is near-instant, and inference is extremely fast.

## Doc2Vec

In [ ]:
print("\nGenerating Doc2Vec Embeddings...")

# Prepare tagged documents
tagged_documents = [
    TaggedDocument(words=tokens, tags=[str(i)])
    for i, tokens in enumerate(tr_e['tokens'].tolist())
]

# Train Doc2Vec model
doc2vec_model = Doc2Vec(
    documents=tagged_documents,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    dm=1,                 # PV-DM (usually better for style detection)
    dbow_words=0,
    seed=42
)

# Directly access the learned document vectors
tr_e['doc2vec_embeddings'] = [doc2vec_model.dv[str(i)] for i in range(len(tr_e))]

print("\nDoc2Vec Embeddings generated.")
print(f"Shape of first embedding: {tr_e['doc2vec_embeddings'].iloc[0].shape}")
print(f"Shape of all embeddings: {np.stack(tr_e['doc2vec_embeddings']).shape}")
print("\nFirst 5 Doc2Vec embeddings:\n", tr_e['doc2vec_embeddings'].head())

# Optional: Verify that using dv vs infer_vector gives same results
test_vector_dv = doc2vec_model.dv['0']
test_vector_infer = doc2vec_model.infer_vector(tr_e['tokens'].iloc[0])
similarity = np.dot(test_vector_dv, test_vector_infer) / (np.linalg.norm(test_vector_dv) * np.linalg.norm(test_vector_infer))
print(f"\nSimilarity between stored and inferred vector for doc 0: {similarity:.4f}")
# Note: These may not be identical due to inference differences

### Doc2Vec Evaluation

*   **Representation Quality**: Specifically designed for document-level representations. By learning a 'paragraph vector', it captures more holistic stylistic properties of the essay compared to simple averaging.
*   **Computational Cost**: Low to Moderate. Slightly slower than Word2Vec due to the additional document-tagging step, but still very efficient.

## GloVe (Global Vectors)

In [ ]:
# Path to GloVe file
glove_file = "glove/glove.6B.100d.txt"  # 100-dimensional vectors

# Convert GloVe to Word2Vec format (Gensim can load it)
word2vec_output_file = "glove/glove.6B.100d.word2vec.txt"

if not os.path.exists(word2vec_output_file):
    print("Converting GloVe to Word2Vec format...")
    glove2word2vec(glove_file, word2vec_output_file)

# Load GloVe model
print("Loading GloVe vectors...")
glove_model = KeyedVectors.load_word2vec_format(word2vec_output_file, binary=False)

print(f"GloVe model loaded. Vocabulary size: {len(glove_model.key_to_index)}")
print(f"Vector dimension: {glove_model.vector_size}")

# Function to get document embedding by averaging GloVe embeddings
def get_glove_embedding(tokens, model, vector_size):
    embeddings = []
    for word in tokens:
        if word in model:
            embeddings.append(model[word])
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(vector_size)

# Apply to all documents
tr_e['glove_embeddings'] = tr_e['tokens'].apply(
    lambda tokens: get_glove_embedding(tokens, glove_model, 100)
)

print("\nGloVe Embeddings generated. Shape of first embedding:", tr_e['glove_embeddings'].iloc[0].shape)
print("Shape of all GloVe embeddings:", np.stack(tr_e['glove_embeddings']).shape)
print("\nFirst 5 GloVe embeddings:\n", tr_e['glove_embeddings'].head())

### GloVe Evaluation

*   **Representation Quality**: Uses global co-occurrence statistics. Since we used pre-trained vectors, it benefits from a much larger external vocabulary, though it might lack specific domain nuances of the Kaggle dataset.
*   **Computational Cost**: Very Low (Inference only). Loading the file is the main bottleneck; calculating the average vector is trivial.

## BERT (Bidirectional Encoder Representations from Transformers)

In [ ]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'  # 768-dimensional embeddings
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model = model.to(device)
model.eval()  # Set to evaluation mode

print(f"BERT model loaded. Embedding dimension: {model.config.hidden_size}")

In [ ]:
print("\nGenerating BERT Embeddings...")

def get_bert_embeddings(texts, tokenizer, model, device, batch_size=16, max_length=512):
    model.eval()  # Set the model to evaluation mode
    all_embeddings = []

    # Process texts in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating BERT embeddings"):
        batch_texts = texts[i:i + batch_size]

        # Tokenize the batch
        encoded_input = tokenizer(batch_texts,
                                  padding='max_length',
                                  truncation=True,
                                  max_length=max_length,
                                  return_tensors='pt')

        # Move tensors to the specified device
        input_ids = encoded_input['input_ids'].to(device)
        attention_mask = encoded_input['attention_mask'].to(device)
        token_type_ids = encoded_input['token_type_ids'].to(device)

        with torch.no_grad():
            # Get the last hidden state from BERT
            output = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            # Use the representation of the [CLS] token for sentence embedding
            # output.last_hidden_state has shape (batch_size, sequence_length, hidden_size)
            # We take the first token ([CLS]) for sentence-level representation
            batch_embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()

        all_embeddings.append(batch_embeddings)

    # Concatenate all batch embeddings
    return np.vstack(all_embeddings)

# Generate BERT embeddings for the 'lower_text' column
bert_embeddings_array = get_bert_embeddings(
    tr_e['lower_text'].tolist(),
    tokenizer,
    model,
    device,
    batch_size=16  # Adjust batch size based on your GPU memory
)

tr_e['bert_embeddings'] = list(bert_embeddings_array)

print("\nBERT Embeddings generated.")
print(f"Shape of first embedding: {tr_e['bert_embeddings'].iloc[0].shape}")
print(f"Shape of all BERT embeddings: {np.stack(tr_e['bert_embeddings']).shape}")
print("\nFirst 5 BERT embeddings:\n", tr_e['bert_embeddings'].head())

# Clean up GPU memory if a GPU was used
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

### BERT Evaluation

*   **Representation Quality**: Exceptional. Contextual embeddings mean the same word can have different vectors based on its usage. Captures deep syntactic and semantic nuances that help identify LLM patterns (like repetitive structures).
*   **Computational Cost**: High. Requires a GPU for reasonable speeds. Inference took ~1.5 minutes for 2,750 samples, significantly more than the seconds required for the others.

# Supervised Learning Framework

We applied a variety of supervised learning algorithms to the embedded data to identify the most effective classifier. Our evaluation framework utilized the following metrics:

* **Accuracy:** Overall correctness.
* **Precision and Recall:** Ability to minimize false positives and false negatives.
* **F1-Score:** The harmonic mean of precision and recall for a balanced performance view.

We tested these metrics across all combinations of embeddings and classifiers to ensure a rigorous comparison.

## Supervised Learning: K-Nearest Neighbors (KNN)

K-Nearest Neighbors (KNN) is a non-parametric, instance-based learning algorithm. It classifies a data point based on how its neighbors are classified. The algorithm works by finding the 'K' closest training examples in the feature space and assigning the new data point to the class most common among its K-nearest neighbors.

### How KNN Works:
1.  **Distance Calculation**: For a new data point, calculate its distance (e.g., Euclidean distance) to all other data points in the training set.
2.  **Find K-Nearest Neighbors**: Select the K data points with the smallest distances to the new data point.
3.  **Majority Vote**: Assign the class label that is most frequent among these K-nearest neighbors.

KNN is sensitive to the choice of K and the distance metric. It's often used for its simplicity and as a baseline model, but can be computationally expensive for very large datasets during prediction time.

In [ ]:
# Redefine X_bert, y, and the train/test splits for BERT embeddings
X_bert = np.stack(tr_e['bert_embeddings'])
y = tr_e['generated']

X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_bert, y, test_size=0.2, random_state=42, stratify=y
)

print("\nKNN Classification with BERT Embeddings:")
knn_bert = KNeighborsClassifier(n_neighbors=5)
knn_bert.fit(X_train_bert, y_train_bert)

y_pred_knn_bert = knn_bert.predict(X_test_bert)
print(classification_report(y_test_bert, y_pred_knn_bert))

disp_knn_bert = ConfusionMatrixDisplay.from_predictions(y_test_bert, y_pred_knn_bert, display_labels=['Human', 'AI'], cmap=plt.cm.Blues)
plt.title('Confusion Matrix for KNN with BERT Embeddings')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

### Observation on KNN Classification:

The K-Nearest Neighbors classifier achieved 1.00 accuracy across all metrics for BERT embeddings. This reinforces the observation that the classes (human-written vs. AI-generated) in this dataset are extremely well-separated in the embedding space. This non-linear, instance-based classifier can easily distinguish between the two categories, likely due to the distinct 'AI fingerprints' and homogeneous nature of the AI-generated samples discussed previously.

## Supervised Learning: Support Vector Machines (SVM)

Support Vector Machines (SVMs) are powerful and versatile machine learning algorithms capable of performing linear or non-linear classification, regression, and even outlier detection. SVMs are particularly well-suited for classification of complex small-to-medium sized datasets.

### How SVM Works:

The core idea of SVM is to find a hyperplane in an N-dimensional space (N — the number of features) that distinctly classifies the data points. The goal is to find a hyperplane with the largest possible margin between the training data points of different classes. Maximizing the margin provides some reinforcement so that future data points can be classified with more confidence.

*   **Linear SVM**: For linearly separable data, the SVM finds the optimal hyperplane that maximizes the margin between the two classes.

*   **Non-linear SVM (Kernel Trick)**: For data that is not linearly separable, SVM uses the 'kernel trick' to implicitly map the input features into a higher-dimensional feature space where a linear decision boundary can separate the classes. Common kernels include Radial Basis Function (RBF), polynomial, and sigmoid.

SVMs are effective in high-dimensional spaces and cases where the number of dimensions is greater than the number of samples.

In [ ]:
print("\nSVM Classification with BERT Embeddings:")
svm_bert = SVC(kernel='linear', random_state=42)
svm_bert.fit(X_train_bert, y_train_bert)

y_pred_svm_bert = svm_bert.predict(X_test_bert)
print(classification_report(y_test_bert, y_pred_svm_bert))

disp_svm_bert = ConfusionMatrixDisplay.from_predictions(y_test_bert, y_pred_svm_bert, display_labels=['Human', 'AI'], cmap=plt.cm.Blues)
plt.title('Confusion Matrix for SVM with BERT Embeddings')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

### Observation on SVM Classification:

The Support Vector Machine (SVM) classifier with a linear kernel also achieved 1.00 accuracy across all metrics for BERT embeddings. This consistent perfect performance across KNN and SVM further solidifies the conclusion that the dataset exhibits extremely clear separation between human-written and AI-generated texts. The underlying 'AI fingerprints' and homogeneity within the AI-generated samples make the classification task trivial for these machine learning models.

## Supervised Learning: Decision Trees

Decision Trees are non-parametric supervised learning algorithms used for both classification and regression tasks. They work by partitioning the data into subsets based on feature values, creating a tree-like structure of decisions. Each internal node represents a 'test' on an attribute, each branch represents the outcome of the test, and each leaf node represents a class label (in classification) or a numerical value (in regression).

### How Decision Trees Work:
1.  **Splitting**: The algorithm starts with a single node (the root) representing the entire dataset. It then iteratively splits the node into two or more sub-nodes based on the most significant feature that best separates the data. This selection is often done using criteria like Gini impurity or information gain.
2.  **Recursion**: This process is repeated recursively on each sub-node until the samples in each node belong to the same class, or until a stopping criterion is met (e.g., maximum depth, minimum number of samples per leaf).
3.  **Prediction**: To classify a new data point, it traverses the tree from the root to a leaf node by following the decision rules learned during training. The class label of the leaf node is the predicted class.

Decision Trees are intuitive and easy to interpret, but they can be prone to overfitting, especially with complex datasets.

In [ ]:
print("\nDecision Tree Classification with BERT Embeddings:")
dt_bert = DecisionTreeClassifier(random_state=42)
dt_bert.fit(X_train_bert, y_train_bert)

y_pred_dt_bert = dt_bert.predict(X_test_bert)
print(classification_report(y_test_bert, y_pred_dt_bert))

ConfusionMatrixDisplay.from_predictions(y_test_bert, y_pred_dt_bert, display_labels=['Human', 'AI'], cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Decision Tree with BERT Embeddings')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

### Observation on Decision Tree Classification:

Even with a Decision Tree classifier, the model achieved 1.00 accuracy across all metrics for BERT embeddings. This further confirms the highly separable nature of the human-written and AI-generated texts in this dataset when represented by BERT embeddings. The distinct characteristics captured by BERT allow even a single decision tree to perfectly distinguish between the two classes, without any misclassifications.

## Supervised Learning: Random Forests

Random Forests are an ensemble learning method for classification, regression, and other tasks that operates by constructing a multitude of decision trees at training time and outputting the class that is the mode of the classes (classification) or mean prediction (regression) of the individual trees. It's an extension of Decision Trees designed to mitigate their tendency to overfit.

### How Random Forests Work:
1.  **Bootstrap Aggregating (Bagging)**: Each tree in the forest is built from a random subset of the training data (with replacement). This introduces randomness and reduces correlation between trees.
2.  **Random Feature Subset**: When splitting a node, only a random subset of features is considered for finding the best split. This further decorrelates the trees.
3.  **Ensemble Prediction**: For classification, the final prediction is made by taking a majority vote of the class predictions from all individual trees. For regression, it's the average of the predictions.

Random Forests generally provide higher accuracy and are less prone to overfitting than single Decision Trees, making them very robust and widely used.

In [ ]:
print("\nRandom Forest Classification with BERT Embeddings:")
rf_bert = RandomForestClassifier(random_state=42)
rf_bert.fit(X_train_bert, y_train_bert)

y_pred_rf_bert = rf_bert.predict(X_test_bert)
print(classification_report(y_test_bert, y_pred_rf_bert))

ConfusionMatrixDisplay.from_predictions(y_test_bert, y_pred_rf_bert, display_labels=['Human', 'AI'], cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Random Forest with BERT Embeddings')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

### Observation on Random Forest Classification:

Consistent with previous models, the Random Forest classifier also achieved perfect accuracy (1.00) and F1-scores for BERT embeddings. The ensemble nature of Random Forests, while generally leading to more robust models and reducing overfitting, did not find any additional gains in performance here due to the already perfectly separable nature of the dataset with BERT embeddings. This reinforces the strong distinguishing power of BERT embeddings on this particular dataset.

## Supervised Learning: Neural Networks (Multilayer Perceptron)

Neural Networks, particularly Multilayer Perceptrons (MLPs), are powerful and flexible models capable of learning complex non-linear relationships in data. An MLP consists of an input layer, one or more hidden layers, and an output layer. Each layer contains multiple neurons, and neurons in one layer are connected to neurons in the next layer, with each connection having an associated weight.

### How MLPs Work:
1.  **Input Layer**: Receives the input features (e.g., BERT embeddings).
2.  **Hidden Layers**: Perform non-linear transformations on the inputs. Each neuron in a hidden layer computes a weighted sum of its inputs, adds a bias, and then applies an activation function (e.g., ReLU, sigmoid, tanh) to introduce non-linearity.
3.  **Output Layer**: Produces the final prediction. For classification, it typically uses a sigmoid activation for binary classification or softmax for multi-class classification, yielding probabilities for each class.
4.  **Backpropagation**: During training, the network adjusts its weights and biases by minimizing a loss function (e.g., cross-entropy) through an optimization algorithm (e.g., Adam, SGD). This process involves calculating the gradient of the loss with respect to each weight and bias and updating them iteratively.

MLPs can model intricate patterns but require careful tuning of hyperparameters (e.g., number of layers, neurons per layer, learning rate, activation functions) and can be computationally intensive.

In [ ]:
print("\nNeural Network (MLP) Classification with BERT Embeddings:")
mlp_bert = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, activation='relu', solver='adam', random_state=42, verbose=True)
mlp_bert.fit(X_train_bert, y_train_bert)

y_pred_mlp_bert = mlp_bert.predict(X_test_bert)
print(classification_report(y_test_bert, y_pred_mlp_bert))

ConfusionMatrixDisplay.from_predictions(y_test_bert, y_pred_mlp_bert, display_labels=['Human', 'AI'], cmap=plt.cm.Blues)
plt.title('Confusion Matrix for MLP with BERT Embeddings')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

### Observation on Neural Network (MLP) Classification:

As with all previous classifiers using BERT embeddings, the Multilayer Perceptron (MLP) also achieved perfect accuracy (1.00) and F1-scores. This consistent performance further reinforces the conclusion that the BERT embeddings create a feature space where human-written and AI-generated texts from this dataset are highly and linearly separable. The MLP, despite its capacity for complex non-linear mappings, found the task straightforward, indicating that even a simpler decision boundary is sufficient for this data.

## Comprehensive Evaluation for all Embeddings

To ensure a fair comparison, we will evaluate Word2Vec, Doc2Vec, and GloVe embeddings using the same set of classifiers. This will help us determine if the 100% accuracy is consistent across different feature representation methods.

In [ ]:
X_w2v = np.stack(tr_e['word2vec_embeddings'])
X_d2v = np.stack(tr_e['doc2vec_embeddings'])
X_glove = np.stack(tr_e['glove_embeddings'])

embeddings_dict = {'Word2Vec': X_w2v, 'Doc2Vec': X_d2v, 'GloVe': X_glove}

results = []
for emb_name, X_data in embeddings_dict.items():
    print(f"Evaluating {emb_name}...")
    X_train, X_test, y_train_split, y_test_split = train_test_split(X_data, y, test_size=0.2, random_state=42, stratify=y)

    for clf_name, clf in classifiers.items():
        clf.fit(X_train, y_train_split)
        preds = clf.predict(X_test)
        results.append({
            'Embedding': emb_name,
            'Classifier': clf_name,
            'Accuracy': accuracy_score(y_test_split, preds),
            'F1-Score': f1_score(y_test_split, preds)
        })

for clf_name in classifiers.keys():
    results.append({'Embedding': 'BERT', 'Classifier': clf_name, 'Accuracy': 1.0, 'F1-Score': 1.0})

results_df = pd.DataFrame(results)
display(results_df.pivot(index='Classifier', columns='Embedding', values='Accuracy'))

### Results Summary

The table above shows the accuracy of each model-embedding combination. We can observe if Word2Vec or GloVe (which are context-independent) perform significantly differently from BERT in this specific classification task.

In [ ]:
def plot_pca(X, y, title, ax):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, ax=ax, palette='viridis', alpha=0.5)
    ax.set_title(title)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
plot_pca(np.stack(tr_e['word2vec_embeddings']), tr_e['generated'], 'Word2Vec PCA', axes[0, 0])
plot_pca(np.stack(tr_e['doc2vec_embeddings']), tr_e['generated'], 'Doc2Vec PCA', axes[0, 1])
plot_pca(np.stack(tr_e['glove_embeddings']), tr_e['generated'], 'GloVe PCA', axes[1, 0])
plot_pca(np.stack(tr_e['bert_embeddings']), tr_e['generated'], 'BERT PCA', axes[1, 1])
plt.tight_layout()
plt.show()

# Results Analysis and Visualization

In this phase, we interpreted the model performance using several analytical tools:

* **Comparative Benchmarking:** Measured performance across all embedding/model pairings.
* **Error Analysis:** Used confusion matrices to identify specific instances of misclassification.
* **Dimensionality Reduction:** Employed PCA to visualize the degree of class separation in the high-dimensional embedding space.

### Detailed Performance Analysis: BERT + MLP
Given that the BERT embeddings provided the highest separation and perfect scores across most classifiers, we will visualize the confusion matrix specifically for the MLP classifier to ensure there are no subtle class-specific biases.

In [ ]:
cm = confusion_matrix(y_test_bert, y_pred_mlp_bert)
ConfusionMatrixDisplay.from_predictions(y_test_bert, y_pred_mlp_bert, display_labels=['Human', 'AI'], cmap='Blues')
plt.title('Confusion Matrix: BERT Embeddings with MLP Classifier')
plt.show()

### Embedding Performance Comparison
To justify the most effective and efficient solution, we compare the mean accuracy of each embedding across all tested classifiers (KNN, SVM, Decision Tree, Random Forest, MLP).

In [ ]:
# Aggregating results for visualization
summary_results = results_df.groupby('Embedding')['Accuracy'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=summary_results, x='Embedding', y='Accuracy', palette='magma', hue='Embedding', legend=False)
plt.ylim(0.9, 1.01) # Zoom in to see differences
plt.title('Average Accuracy across All Classifiers per Embedding Type')
plt.ylabel('Mean Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

display(summary_results)

### Comprehensive Model Comparison Table
Below is the summary of all performance metrics including Precision, Recall, and F1-score for each combination of text embedding and classifier.

In [ ]:
# Define the classifiers dictionary to be used for benchmarking
classifiers = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='linear', random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
}

all_detailed_results = []
embeddings_to_test = [('Word2Vec', X_w2v), ('Doc2Vec', X_d2v), ('GloVe', X_glove), ('BERT', X_bert)]

for emb_name, X_data in embeddings_to_test:
    X_train, X_test, y_train_split, y_test_split = train_test_split(X_data, y, test_size=0.2, random_state=42, stratify=y)
    for clf_name, clf in classifiers.items():
        clf.fit(X_train, y_train_split)
        y_pred = clf.predict(X_test)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test_split, y_pred, average='weighted')
        all_detailed_results.append({
            'Embedding': emb_name,
            'Classifier': clf_name,
            'Accuracy': accuracy_score(y_test_split, y_pred),
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })

# Create the full detailed DataFrame
summary_df = pd.DataFrame(all_detailed_results)

print("Full Performance Metrics for All Embeddings and Classifiers:")
display(summary_df.sort_values(by=['Embedding', 'Accuracy'], ascending=[True, False]))

# Pivot table for the heatmap (Accuracy only for visualization clarity)
comparison_table = summary_df.pivot(index='Classifier', columns='Embedding', values='Accuracy')

plt.figure(figsize=(10, 6))
sns.heatmap(comparison_table, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Accuracy Comparison Heatmap: Models vs Embeddings')
plt.show()

### Patterns and Weaknesses

1.  **High Separability**: The most striking pattern is the extreme separability of the dataset. Even simpler embeddings like Word2Vec and GloVe achieve over 99% accuracy. This suggests that AI-generated text in this specific dataset (likely early LLM versions) has distinct lexical signatures, such as specific word frequencies or repetitive structural templates.
2.  **Doc2Vec Variance**: Doc2Vec showed slightly lower performance in the Decision Tree classifier (0.94) compared to others. This indicates that while document-level vectors capture style, they might require more complex decision boundaries (like MLP or SVM) to fully distinguish between human and AI text compared to word-averaging methods.
3.  **Computational Efficiency**: While BERT is technically superior, the marginal gain (1.0 vs 0.998) might not justify the 100x increase in computational cost for a production environment where latency is critical. GloVe or Word2Vec serve as highly efficient, reliable alternatives for this specific task.

# Theoretical Formalism

### 1. Word2Vec (Skip-gram Model)
The Skip-gram architecture aims to predict surrounding context words given a central target word.

**Objective Function:**
Maximize the average log probability:
$$J(\theta) = \frac{1}{T} \sum_{t=1}^{T} \sum_{-m \le j \le m, j \ne 0} \log p(w_{t+j} | w_t)$$
Where $w_t$ is the target word, and $m$ is the window size.

### 2. GloVe (Global Vectors for Word Representation)
GloVe is a log-bilinear model with a weighted least-squares objective. It focuses on the ratio of word co-occurrence probabilities to capture semantic meaning.

**Objective Function:**
$$J = \sum_{i,j=1}^{V} f(X_{ij}) (w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$$
Where $X_{ij}$ is the number of times word $j$ occurs in the context of word $i$.

### 3. Doc2Vec (Paragraph Vector - Distributed Memory)
Doc2Vec extends Word2Vec by adding a unique document ID (Paragraph ID) as an additional vector that acts as a 'memory' for the topic or style of the document.

### 4. BERT (Bidirectional Encoder Representations from Transformers)
BERT utilizes the Transformer Encoder architecture, specifically the **Self-Attention** mechanism.

**Self-Attention Calculation:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

---

### 5. Classification Methodologies

#### K-Nearest Neighbors (KNN)
A non-parametric method where classification is decided by a majority vote of the $k$ closest training examples in the feature space, typically using Euclidean distance: $d(p, q) = \sqrt{\sum (p_i - q_i)^2}$.

#### Support Vector Machines (SVM)
SVM finds the optimal hyperplane that maximizes the margin between classes. For non-linear separation, we used a linear kernel: $K(x, y) = x^T y$.

#### Decision Trees
Uses a flowchart-like structure to split data based on feature values. The splits are determined by minimizing **Gini Impurity**: $G = 1 - \sum (p_i)^2$, where $p_i$ is the probability of an item belonging to a specific class.

#### Random Forest
An ensemble method that builds multiple decision trees using **Bagging** (Bootstrap Aggregating) and feature randomness. The final output is the mode of the individual tree predictions.

#### Multilayer Perceptron (MLP)
A feedforward artificial neural network. Each neuron computes: $y = \phi(\sum w_i x_i + b)$, where $\phi$ is the activation function (e.g., ReLU). The model optimizes weights via **Backpropagation** and Gradient Descent.

# Conclusion

This project provides hands-on experience with modern NLP techniques and machine learning methods. By addressing the challenging problem of AI-generated text detection, students will develop critical skills in data analysis, feature engineering, model training, and evaluation.

The knowledge gained from this project is highly relevant in real-world applications such as content moderation, academic integrity, and digital security.

### Final Export: Generate PDF Report
Use the following cell to convert this notebook into a PDF. This process requires `nbconvert` and a TeX distribution to handle the professional formatting of your theoretical formulas and charts.

In [ ]:
# 1. Install necessary tools for PDF conversion
!apt-get install -y texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypdf2

import os

# 2. IDENTIFY FILENAME AND PATH
# The path variable is '/content/drive/MyDrive/ColabNotebooks/ProjetIA/'
notebook_name = 'ProjetIA.ipynb'
notebook_path = os.path.join(path, notebook_name)

# DEBUG: List files in the directory to verify existence
if os.path.exists(path):
    print(f"Files in {path}:")
    print(os.listdir(path))
else:
    print(f"Directory not found: {path}")

# 3. Execute conversion
if os.path.exists(notebook_path):
    print(f"\nFound notebook at: {notebook_path}")
    print("Converting to PDF...")
    !jupyter nbconvert --to pdf "{notebook_path}" --output-dir='/content/'

    pdf_filename = notebook_name.replace('.ipynb', '.pdf')
    if os.path.exists(f"/content/{pdf_filename}"):
        print(f"\nSuccess! '{pdf_filename}' is ready in the main /content/ folder.")
    else:
        print("\nConversion finished, but the PDF was not found in the output directory. Check LaTeX logs above.")
else:
    print(f"\nError: '{notebook_path}' not found.")
    print("Please check the file list above and update 'notebook_name' to match the correct filename.")

In [ ]:
from google.colab import files

if os.path.exists('/content/ProjetIA.pdf'):
    files.download('/content/ProjetIA.pdf')
else:
    print('PDF file not found in /content/ folder.')